# Single-Object Test — `Dino_3` (dinosaur)

Runs **all four engines** on exactly **one** GSO object: `Dino_3`, the dinosaur
figurine from the canonical-10 subset. Everything else is untouched.

**Why this object:** it is in the canonical-10 subset, so complete published reference
values exist for all four engines. That makes this a *correctness* test, not just a
smoke test — the final cell diffs your run against the published 400-run table.

**What it measures beyond the paper:**
- `t_load` — wall-clock model load time per engine (the paper never reports this)
- `t_infer` — pure forward pass, comparable to the published `latency_s`
- `t_e2e` — full runner time for one object (includes alignment + PSNR/LPIPS)

**Expected runtime:** ~25-35 min. SPAR3D alone is ~38 s inference plus several
minutes of CUDA extension compilation on first run.


In [ ]:
# --- Error logging setup ---
import sys, traceback, datetime
from pathlib import Path
LOG_FILE = Path("/kaggle/working/outputs/benchmark_log.txt")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()
log_f = open(LOG_FILE, "a")
sys.stdout = Tee(sys.stdout, log_f)
sys.stderr = Tee(sys.stderr, log_f)
print(f"=== BENCHMARK LOG START {datetime.datetime.now()} ===")


In [ ]:
import os, subprocess, sys, glob, gc, json
from pathlib import Path

# 1. Headless OpenGL & T4 CUDA compilation flags (sm_75 only -> 10x faster C++/CUDA builds!)
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["SPCONV_DISABLE_JIT"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "4"

# 2. HuggingFace token check
_hf_token = None
try:
    for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
        if os.environ.get(_k):
            _hf_token = os.environ[_k]
            print(f"HF_TOKEN found in environment via {_k} ✅")
            break
    if not _hf_token:
        from kaggle_secrets import UserSecretsClient
        _client = UserSecretsClient()
        for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
            try:
                _val = _client.get_secret(_k)
                if _val:
                    _hf_token = _val
                    print(f"HF_TOKEN found in Kaggle Secrets under '{_k}' ✅")
                    break
            except Exception:
                pass
except Exception as _e:
    print(f"Notice: Kaggle secrets check: {type(_e).__name__}: {_e}")

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    os.environ["HUGGINGFACE_TOKEN"] = _hf_token
    try:
        from huggingface_hub import login
        login(token=_hf_token, write_to_disk=False)
    except Exception:
        pass
    print("HF_TOKEN configured ✅")
else:
    print("Notice: HF_TOKEN not attached (public/mirror models will still work).")

import torch
print(f"torch {torch.__version__} (CUDA: {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU attached! Enable Accelerator > GPU T4 x2 in kernel settings.")
props = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {props.name} | {props.total_memory/1e9:.1f} GB VRAM")

# ---- THE ONLY THING THAT CHANGES vs the 100-object run: ONE OBJECT -----------
TARGET_OBJECTS = ['Dino_3']
NUM_SAMPLES = len(TARGET_OBJECTS)          # 1 — the runner sizes everything from this
assert NUM_SAMPLES == 1, "This notebook is deliberately single-object."

GSO_DIR = "/kaggle/working/data/gso"
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)

import numpy
open("/tmp/constraints.txt", "w").write(f"numpy=={numpy.__version__}\n")
print(f"Config: NUM_SAMPLES={NUM_SAMPLES} | object={TARGET_OBJECTS[0]} | numpy pinned: {numpy.__version__}")


In [ ]:
import os, sys, subprocess, datetime
import numpy as _np
if not os.path.exists("/tmp/constraints.txt"):
    open("/tmp/constraints.txt", "w").write(f"numpy=={_np.__version__}\n")

# Install pure wheels respecting Kaggle numpy==2.1.3 constraint, then install
# lpips/rembg/pyrender with --no-deps so rembg>=2.0.85 (which pins numpy>=2.3.0)
# never forces pip to backtrack scikit-image/scipy to a 22.7 MB source tarball.
!pip install -q -c /tmp/constraints.txt trimesh==4.4.1 onnxruntime==1.20.1 pooch==1.8.2 pymatting==1.1.12 pyglet==2.0.10 "PyOpenGL>=3.1.7"
!pip install -q --no-deps lpips rembg pyrender
!pip install -q freetype-py==2.5.1

import torch
assert torch.cuda.is_available(), "torch lost CUDA during installs"
import lpips
_lpips_vgg = lpips.LPIPS(net="vgg")
del _lpips_vgg
from rembg import new_session
_sess = new_session("isnet-general-use")
print("=== CELL 3 GATE: PASS (LPIPS + ISNet cached) ✅ ===")


In [ ]:
import os, sys, shutil, hashlib, subprocess

CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
shutil.rmtree(CODE_DIR, ignore_errors=True)
_r = subprocess.run(["git", "init", CODE_DIR], capture_output=True, text=True)
if _r.returncode == 0:
    _r = subprocess.run(["git", "-C", CODE_DIR, "remote", "add", "origin", "https://github.com/anubhavaanand/Project-red-2d-to-3d.git"], capture_output=True, text=True)
if _r.returncode == 0:
    # GitHub does not serve `clone -b <sha>` for arbitrary commit SHAs
    # ("remote branch not found"), but it does allow shallow-fetching a
    # specific SHA reachable from a ref. Fetch the pin, check it out.
    _r = subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "1", "origin", "aae25900d34298a5188d9d85ee6394aaa769928c"], capture_output=True, text=True)
if _r.returncode == 0:
    _r = subprocess.run(["git", "-C", CODE_DIR, "checkout", "FETCH_HEAD"], capture_output=True, text=True)
if _r.returncode != 0 or not os.path.exists(os.path.join(CODE_DIR, "src")):
    _msg = ((_r.stdout or "") + (_r.stderr or ""))[-600:]
    raise RuntimeError("GitHub pinned-SHA fetch failed:" + chr(10) + _msg)

_sha = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

for _k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]
from src.evaluation.benchmark_runner import BenchmarkRunner, load_gso_samples, bake_texture_to_vertices
from src.engines.triposr_engine import TriposrEngine
from src.engines.lgm_engine import LGMEngine
from src.engines.trellis_engine import TrellisEngine
from src.engines.spar3d_engine import Spar3dEngine
os.makedirs("/kaggle/working/outputs", exist_ok=True)
with open("/kaggle/working/outputs/code_sha.txt", "w") as _f:
    _f.write(_sha + "\n")
print(f"code cloned fresh @ {_sha[:12]} — all 4 engines imported OK ✅ (sha recorded in outputs/code_sha.txt)")
if _sha != "aae25900d34298a5188d9d85ee6394aaa769928c":
    print(f"WARNING: HEAD {_sha[:12]} != expected pin aae25900d342 — numbers may not match the published table")


In [ ]:
import os, sys, glob, json, zipfile, tempfile, shutil, random
from pathlib import Path
import numpy as np, trimesh
from PIL import Image as PILImage

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
if "GSO_DIR" not in globals():
    GSO_DIR = "/kaggle/working/data/gso"
if "CODE_DIR" not in globals():
    CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.evaluation.benchmark_runner import load_gso_samples, bake_texture_to_vertices, render_gso_mesh_view

def have_all(root):
    root = Path(root)
    for n in TARGET_OBJECTS:
        m = root / n / "meshes"
        if not (m.is_dir() and (list(m.glob("*.obj")) or list(m.glob("*.ply")))):
            return False
    return True

REPO = "suvadityamuk/google-scanned-objects-raw"

if not have_all(GSO_DIR):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    all_zips = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".zip"))
    _zset = set(all_zips)
    _missing = [n for n in TARGET_OBJECTS if (n + ".zip") not in _zset]
    if _missing:
        raise RuntimeError(f"GSO zips missing upstream: {_missing}")
    wanted = [n + ".zip" for n in TARGET_OBJECTS]

    def _download_with_retry(zip_name, attempts=4):
        last_err = None
        for attempt in range(1, attempts + 1):
            try:
                return hf_hub_download(REPO, zip_name, repo_type="dataset", local_dir="/tmp/gso_zips", token=os.environ.get("HF_TOKEN"))
            except Exception as e:
                last_err = e
                print(f"  retry {attempt}/{attempts} for {zip_name}: {str(e)[:80]}")
        raise last_err

    def ingest(zip_name):
        obj_id = zip_name[:-4]
        try:
            zp = _download_with_retry(zip_name)
            with tempfile.TemporaryDirectory() as td:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(td)
                hits = list(Path(td).rglob("*.obj")) + list(Path(td).rglob("*.ply"))
                if not hits:
                    return False
                mdir = Path(GSO_DIR) / obj_id / "meshes"
                mdir.mkdir(parents=True, exist_ok=True)
                for h in hits:
                    shutil.copy(h, mdir / h.name)
                for sib in hits[0].parent.iterdir():
                    if sib.suffix.lower() in (".mtl", ".png", ".jpg", ".jpeg"):
                        shutil.copy(sib, mdir / sib.name)
                for tex in Path(td).rglob("*.png"):
                    if not (mdir / tex.name).exists():
                        shutil.copy(tex, mdir / tex.name)
            return True
        except Exception as e:
            print("  skip", obj_id, str(e)[:80])
            return False

    ok = 0
    for zname in wanted:
        if ingest(zname):
            ok += 1
            print(f"[{ok}/{len(wanted)}] Ingested {zname[:-4]}")
    still_missing = [n for n in TARGET_OBJECTS if not (Path(GSO_DIR) / n / "meshes").is_dir()]
    if still_missing:
        print(f"[repair] retrying {len(still_missing)} missing objects")
        for n in still_missing:
            if ingest(n + ".zip"):
                ok += 1
                print(f"[repair] recovered {n}")
    print(f"ingested {ok}/{len(wanted)} objects into {GSO_DIR}")
    shutil.rmtree("/tmp/gso_zips", ignore_errors=True)

# Render the conditioning view (az=20,el=15 @1024 RGBA) and the GT novel view
# (az=65,el=15 @256 RGB) from the native 4096x4096 texture map.
rendered = 0
for d in sorted(Path(GSO_DIR).iterdir()):
    if not d.is_dir() or d.name not in TARGET_OBJECTS:
        continue
    mcands = list((d/("meshes")).glob("*.obj")) + list((d/("meshes")).glob("*.ply"))
    if not mcands or (d/("images")/".upright_v6").exists():
        continue
    try:
        (d/("images")).mkdir(parents=True, exist_ok=True)
        rgba_000 = render_gso_mesh_view(mcands[0], azimuth_deg=20.0, elevation_deg=15.0, resolution=1024, return_rgba=True)
        rgb_001 = render_gso_mesh_view(mcands[0], azimuth_deg=65.0, elevation_deg=15.0, resolution=256, return_rgba=False)
        if rgba_000 is None or rgb_001 is None:
            continue
        PILImage.fromarray(rgba_000, mode="RGBA").save(d/("images")/"000.png")
        PILImage.fromarray(rgb_001).save(d/("images")/"001.png")
        (d/("images")/".upright_v6").write_text("ok")
        rendered += 1
        print(f"rendered views for {d.name}")
    except Exception as e:
        print("render failed:", d.name, str(e)[:100])

samples = load_gso_samples(GSO_DIR, num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS)
_missing = [o for o in TARGET_OBJECTS if o not in {sm.sample_id for sm in samples}]
if _missing or len(samples) != len(TARGET_OBJECTS):
    raise RuntimeError(f"GSO pre-flight failed: {len(samples)}/{len(TARGET_OBJECTS)} ready, missing={_missing}")
print(f"GSO ready: rendered {rendered} view(s) | loaded {len(samples)}/{len(TARGET_OBJECTS)} sample(s) ✅")

# --- Brightness sentinel: catch a regressed renderer --------------------------
try:
    from PIL import Image
    import numpy as np
    _img = Path(GSO_DIR) / TARGET_OBJECTS[0] / "images" / "001.png"
    if _img.exists():
        _a = np.asarray(Image.open(_img).convert("RGB"), dtype=np.float32)
        _mask = _a.mean(axis=2) < 240
        _fg_mean = float(_a[_mask].mean()) if _mask.any() else float(_a.mean())
        print(f"[SENTINEL] GT 001.png foreground brightness = {_fg_mean:.1f}")
        assert 60 <= _fg_mean <= 250, f"Renderer regression: brightness {_fg_mean:.1f} implausible"
        print("[SENTINEL] PASS ✅")
except Exception as _e:
    print(f"[SENTINEL] Could not verify brightness: {_e}")


In [ ]:
# --- Timing helper: separates model load from inference ----------------------
# The runner calls engine.load_model() itself only if the model is not already
# loaded (benchmark_runner.py:1029-1030), so loading here first means:
#   t_load  = our measured load      (never reported in the paper)
#   t_infer = engine-reported latency (comparable to published latency_s)
#   t_e2e   = full runner wall clock  (adds alignment + PSNR/LPIPS)
import time, gc, torch, datetime

TIMINGS = []   # one dict per engine

def load_and_time(engine, label):
    """Load one engine, recording wall-clock load time and peak VRAM."""
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    engine.load_model()
    dt = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() / (1024**3)
    rec = {"engine": label, "t_load_s": round(dt, 2), "vram_after_load_gb": round(peak, 2)}
    TIMINGS.append(rec)
    print(f"[{label}] model load: {dt:.1f}s | VRAM after load: {peak:.2f} GB")
    return engine


def run_one(engine, label, out_dir):
    """Run the benchmark on the single object and record end-to-end time."""
    runner = BenchmarkRunner(
        dataset_path=GSO_DIR,
        engines=[engine],
        num_samples=NUM_SAMPLES,
        object_ids=TARGET_OBJECTS,
        output_dir=out_dir,
        resume_json_path=None,
        save_mesh_limit=1,          # one object -> one mesh
        max_wall_minutes=None,
    )
    t0 = time.perf_counter()
    print(f"[{label}] {datetime.datetime.now().strftime('%H:%M:%S')} starting run...")
    df = runner.run()
    e2e = time.perf_counter() - t0
    print(f"[{label}] {datetime.datetime.now().strftime('%H:%M:%S')} COMPLETE ✅  e2e={e2e:.1f}s")

    # Merge e2e into the timing record for this engine.
    for rec in TIMINGS:
        if rec["engine"] == label:
            rec["t_e2e_s"] = round(e2e, 2)
    runner.save_results(f"/kaggle/working/outputs/{label.lower()}_results.json")
    return runner, df

print("timing helpers ready")


In [ ]:
# --- TripoSR (Triplane NeRF -> 256^3 Marching Cubes) ---
import os, sys, gc, datetime, pathlib, subprocess, traceback
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    if not os.path.isdir("/tmp/TripoSR"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/VAST-AI-Research/TripoSR.git", "/tmp/TripoSR"], check=True)
    subprocess.run(["git", "-C", "/tmp/TripoSR", "fetch", "--depth", "1", "origin", "107cefdc244c39106fa830359024f6a2f1c78871"], check=True)
    subprocess.run(["git", "-C", "/tmp/TripoSR", "checkout", "FETCH_HEAD"], check=True)
    print("[pin] VAST-AI-Research/TripoSR @ 107cefdc24 OK")
    if "/tmp/TripoSR" not in sys.path:
        sys.path.insert(0, "/tmp/TripoSR")

        # --- KERNEL PATCH: accept the engine's revision= kwarg upstream ---------
        # TSR.from_pretrained is a custom classmethod without **kwargs; the engine
        # (frozen at the benchmark pin) passes revision=<sha>. Rewrite upstream to
        # accept it and forward to hf_hub_download, which supports commit SHAs.
        _tsr_sys = "/tmp/TripoSR/tsr/system.py"
        _tsr_src = open(_tsr_sys).read()
        if "revision: str | None = None" not in _tsr_src:
            _tsr_new = _tsr_src.replace(
                "cls, pretrained_model_name_or_path: str, config_name: str, weight_name: str",
                "cls, pretrained_model_name_or_path: str, config_name: str, weight_name: str, revision: str | None = None",
            ).replace(
                "repo_id=pretrained_model_name_or_path, filename=config_name",
                "repo_id=pretrained_model_name_or_path, filename=config_name, revision=revision",
            ).replace(
                "repo_id=pretrained_model_name_or_path, filename=weight_name",
                "repo_id=pretrained_model_name_or_path, filename=weight_name, revision=revision",
            )
            if _tsr_new == _tsr_src:
                print("[TripoSR] WARNING: system.py patch anchors not found; revision kwarg will fail")
            else:
                open(_tsr_sys, "w").write(_tsr_new)
                print("[TripoSR] Patched tsr/system.py to accept revision= OK")
        else:
            print("[TripoSR] system.py already patched OK")

    !pip install -q -c /tmp/constraints.txt omegaconf einops pytorch_lightning e3nn transformers

    pathlib.Path("/kaggle/working/torchmcubes.py").write_text("""import numpy as np
import torch
from skimage import measure

def marching_cubes(level, iso=0.0):
    if level.dim() == 4:
        level = level.squeeze(0)
    vol = level.detach().cpu().numpy().astype(np.float32)
    try:
        v, f, _, _ = measure.marching_cubes(vol, float(iso))
        v = np.ascontiguousarray(v[:, [2, 1, 0]], dtype=np.float32)
        f = np.ascontiguousarray(f, dtype=np.int64)
    except Exception:
        return (torch.zeros((0, 3), dtype=torch.float32),
                torch.zeros((0, 3), dtype=torch.int64))
    return torch.from_numpy(v), torch.from_numpy(f)
""")
    if "/kaggle/working" not in sys.path:
        sys.path.insert(0, "/kaggle/working")

    from src.engines.triposr_engine import TriposrEngine

    tr_engine = TriposrEngine(device=("cuda" if torch.cuda.is_available() else "cpu"))
    load_and_time(tr_engine, "TripoSR")
    runner, df = run_one(tr_engine, "TripoSR", "/kaggle/working/outputs/triposr")
    display(df)
    del runner, tr_engine
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
except Exception as e:
    print(f"[TripoSR] Error: {type(e).__name__}: {e}")
    traceback.print_exc()


In [ ]:
# --- LGM (ImageDream 4-View + Asymmetric U-Net -> 65,536 Gaussians -> MC Mesh) ---
import os, sys, gc, glob, datetime, subprocess, traceback
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    !pip install -q -c /tmp/constraints.txt --no-deps "kiui==0.3.4"
    !pip install -q -c /tmp/constraints.txt varname lazy_loader objprint rich pyyaml questionary tyro einops roma plyfile

    for _h in glob.glob("/usr/local/lib/python*/dist-packages/kiui/typing.py"):
        open(_h, "w").write("from typing import Sequence, List, Tuple, Dict, Any, Optional, Union, Literal, Callable\nfrom torch import Tensor\nfrom numpy import ndarray\n")
    for _k in [k for k in sys.modules if k.startswith("kiui")]:
        del sys.modules[_k]

    for _url, _dest in [
        ("https://github.com/ashawkey/mvdream_diffusers.git", "/tmp/mvdream"),
        ("https://github.com/3DTopia/LGM.git", "/tmp/LGM"),
        ("https://github.com/ashawkey/diff-gaussian-rasterization.git", "/tmp/dgr"),
    ]:
        if not os.path.isdir(_dest):
            subprocess.run(["git", "clone", "--recursive", "--depth", "1", _url, _dest], check=True)

            _PIN_MAP = {
    "https://github.com/3DTopia/LGM.git": "fe8d12cff8c827df7bb77a3c8e8b37408cb6fe4c",
    "https://github.com/ashawkey/mvdream_diffusers.git": "8b318efac486784819502b88952fae970f40a47d",
    "https://github.com/ashawkey/diff-gaussian-rasterization.git": "8829d14f814fccdaf840b7b0f3021a616583c0a1",
            }
            _pin = _PIN_MAP.get(_url)
            if _pin:
                subprocess.run(["git", "-C", _dest, "fetch", "--depth", "1", "origin", _pin], check=True)
                subprocess.run(["git", "-C", _dest, "checkout", "FETCH_HEAD"], check=True)
                print("[pin]", _dest, "@", _pin[:10], "OK")
    !pip install -q -c /tmp/constraints.txt --no-build-isolation /tmp/dgr

    for _p in ["/tmp/mvdream", "/tmp/LGM"]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # py3.13 image ships diffusers 0.42, which REFUSES custom model code
    # (mv_unet.py inside the pinned imagedream snapshot) unless
    # trust_remote_code=True. SEC-07 pinned that snapshot to a reviewed commit.
    os.environ["HF_HUB_TRUST_REMOTE_CODE"] = "1"
    try:
        import diffusers as _df
        _orig_fpt = _df.DiffusionPipeline.from_pretrained.__func__
        def _trust_fpt(cls, *a, **kw):
            kw.setdefault("trust_remote_code", True)
            return _orig_fpt(cls, *a, **kw)
        _df.DiffusionPipeline.from_pretrained = classmethod(_trust_fpt)
        print("[LGM] diffusers trust_remote_code patch applied OK")
    except Exception as _e:
        print(f"[LGM] diffusers patch skipped: {type(_e).__name__}: {_e}")

    from src.engines.lgm_engine import LGMEngine

    lg_engine = LGMEngine(device="cuda")
    load_and_time(lg_engine, "LGM")
    runner, df = run_one(lg_engine, "LGM", "/kaggle/working/outputs/lgm")
    display(df)
    del runner, lg_engine
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
except Exception as e:
    print(f"[LGM] Error: {type(e).__name__}: {e}")
    traceback.print_exc()


In [ ]:
# --- SPAR3D (Point-E 512-Pt Diffusion + Triplane DMTet + PBR) ---
# Wrapped in top-level try/except so Papermill (allow_errors=False) never aborts the kernel.
import os, sys, gc, datetime, shutil, subprocess, types, traceback
from pathlib import Path
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["SPAR3D_LOW_VRAM"] = "0"
    os.environ["CC"] = "gcc"
    os.environ["CXX"] = "g++"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    from src.engines.spar3d_engine import Spar3dEngine, _install_spar3d_compat_shims

    _install_spar3d_compat_shims()

    # 1. Install Python deps FIRST (ftfy, regex, loralib) BEFORE importing clip/alpha_clip
    !pip install -q -c /tmp/constraints.txt einops==0.8.0 jaxtyping==0.2.34 omegaconf==2.3.0 loralib==0.1.2 ftfy==6.2.1 regex==2024.9.11 timm==1.0.9 kornia==0.7.3

    # 2. Clone CLIP & AlphaCLIP, patch pkg_resources, add to sys.path
    for _repo_url, _dest in [
        ("https://github.com/openai/CLIP.git", "/tmp/CLIP"),
        ("https://github.com/SunzeY/AlphaCLIP.git", "/tmp/AlphaCLIP"),
    ]:
        if not os.path.isdir(os.path.join(_dest, ".git")):
            shutil.rmtree(_dest, ignore_errors=True)
            subprocess.run(["git", "clone", "--depth", "1", _repo_url, _dest], check=True)

            _PIN_MAP = {
    "https://github.com/openai/CLIP.git": "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6",
    "https://github.com/SunzeY/AlphaCLIP.git": "ef9262bc539728bf8ef2dfe9c402ae12bbfcd9ff",
            }
            _pin = _PIN_MAP.get(_repo_url)
            if _pin:
                subprocess.run(["git", "-C", _dest, "fetch", "--depth", "1", "origin", _pin], check=True)
                subprocess.run(["git", "-C", _dest, "checkout", "FETCH_HEAD"], check=True)
                print("[pin]", _dest, "@", _pin[:10], "OK")
        for _pyf in Path(_dest).rglob("*.py"):
            try:
                _t = _pyf.read_text()
                if "from pkg_resources import packaging" in _t:
                    _pyf.write_text(_t.replace("from pkg_resources import packaging", "import packaging"))
            except Exception:
                pass
        if _dest not in sys.path:
            sys.path.insert(0, _dest)

    import clip, alpha_clip
    print("[SPAR3D] clip and alpha_clip imported cleanly ✅")

    # 3. Clone Stability-AI/stable-point-aware-3d and build sm_75 CUDA extensions
    SPAR3D_DIR = "/tmp/stable-point-aware-3d"
    if not os.path.isdir(os.path.join(SPAR3D_DIR, ".git")):
        shutil.rmtree(SPAR3D_DIR, ignore_errors=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-point-aware-3d.git", SPAR3D_DIR],
            check=True,
        )
    subprocess.run(["git", "-C", "SPAR3D_DIR", "fetch", "--depth", "1", "origin", "fdc311b16809e6a8adc2f5a3407ebb3db1a95bd1"], check=True)
    subprocess.run(["git", "-C", "SPAR3D_DIR", "checkout", "FETCH_HEAD"], check=True)
    print("[pin] Stability-AI/stable-point-aware-3d @ fdc311b168 OK")
    if SPAR3D_DIR not in sys.path:
        sys.path.insert(0, SPAR3D_DIR)

        # --- KERNEL PATCH: accept the engine's revision= kwarg upstream ---------
        _spar_sys = os.path.join(SPAR3D_DIR, "spar3d", "system.py")
        _spar_src = open(_spar_sys).read()
        if "revision: str | None = None" not in _spar_src:
            _spar_new = _spar_src.replace(
                "low_vram_mode: bool = False,",
                "low_vram_mode: bool = False, revision: str | None = None,",
            ).replace(
                "filename=config_name\n            )",
                "filename=config_name, revision=revision\n            )",
            ).replace(
                "filename=weight_name\n            )",
                "filename=weight_name, revision=revision\n            )",
            )
            if _spar_new == _spar_src:
                print("[SPAR3D] WARNING: system.py patch anchors not found; revision kwarg will fail")
            else:
                open(_spar_sys, "w").write(_spar_new)
                print("[SPAR3D] Patched spar3d/system.py to accept revision= OK")
        else:
            print("[SPAR3D] system.py already patched OK")

    try:
        import nvdiffrast.torch  # noqa: F401
    except ImportError:
        !pip install -q -c /tmp/constraints.txt --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

    print("[SPAR3D] Building texture_baker and uv_unwrapper CUDA extensions (sm_75) …")
    _tb = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/texture_baker"],
        capture_output=True, text=True,
    )
    _uw = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/uv_unwrapper"],
        capture_output=True, text=True,
    )
    print(f"[SPAR3D] CUDA extensions: texture_baker={_tb.returncode}, uv_unwrapper={_uw.returncode}")

    sp_engine = Spar3dEngine(device="cuda", texture_resolution=1024, low_vram_mode=False)
    load_and_time(sp_engine, "SPAR3D")
    runner, df = run_one(sp_engine, "SPAR3D", "/kaggle/working/outputs/spar3d")
    display(df)
    del runner, sp_engine
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
except Exception as e:
    print(f"[SPAR3D] Failed with {type(e).__name__}: {e}")
    traceback.print_exc()


In [ ]:
# --- TRELLIS (Sparse Structure Flow + SLAT Rectified Flow -> 256^3 FlexiCubes) ---
import os, sys, gc, glob, json, shutil, subprocess, datetime, types, traceback
from pathlib import Path
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["ATTN_BACKEND"] = "xformers"
    os.environ["SPCONV_ALGO"] = "native"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    !pip install -q -c /tmp/constraints.txt "pybind11==2.13.6" cumm-cu126 spconv-cu126
    _c8incs = sorted(glob.glob("/usr/local/lib/python3*/dist-packages/cumm/include")) + \
              sorted(glob.glob("/opt/conda/lib/python3*/site-packages/cumm/include"))
    if _c8incs:
        os.environ["CPATH"] = _c8incs[0] + (os.pathsep + os.environ["CPATH"] if os.environ.get("CPATH") else "")

    try:
        import xformers  # noqa: F401
    except ImportError:
        # SEC-24: subprocess with an argv list (no shell) and check=True so a
        # failed install is loud.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "-c", "/tmp/constraints.txt", "--no-deps", "xformers"],
            check=True,
        )

    !pip install -q -c /tmp/constraints.txt pytorch-lightning==2.4.0 omegaconf==2.3.0 easydict==1.13 "imageio[ffmpeg]" timm kornia open3d
    try:
        import nvdiffrast.torch  # noqa: F401
    except ImportError:
        !pip install -q -c /tmp/constraints.txt --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

    def _ensure_checkout(repo_url, dest, pin):
        if os.path.isdir(dest) and not os.path.isdir(os.path.join(dest, ".git")):
            shutil.rmtree(dest, ignore_errors=True)
        if not os.path.isdir(os.path.join(dest, ".git")):
            subprocess.run(["git", "clone", "--depth", "1", "--recurse-submodules", repo_url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-f", pin], capture_output=True)

    _ensure_checkout("https://github.com/microsoft/TRELLIS.git", "/tmp/TRELLIS", "442aa1e1afb9014e80681d3bf604e8d728a86ee7")
    if "/tmp/TRELLIS" not in sys.path:
        sys.path.insert(0, "/tmp/TRELLIS")

    # Patch TRELLIS base.py: surface swallowed primary-load errors instead of
    # silently falling back to a bogus HF repo id (bare `v` -> 401).
    _base_py = "/tmp/TRELLIS/trellis/pipelines/base.py"
    _base_src = open(_base_py).read()
    _old = "            except:\n                _models[k] = models.from_pretrained(v)"
    _new = (
        "            except Exception as _primary_err:\n"
        "                print(f\"[TRELLIS-loader] {k}: primary load failed: {type(_primary_err).__name__}: {_primary_err}\")\n"
        "                if os.path.isdir(path):\n"
        "                    raise\n"
        "                _models[k] = models.from_pretrained(v)"
    )
    if _old in _base_src:
        open(_base_py, "w").write(_base_src.replace(_old, _new))
        print("[TRELLIS] Patched base.py to surface primary-load errors ✅")
    elif "[TRELLIS-loader]" in _base_src:
        print("[TRELLIS] base.py already patched ✅")
    else:
        print("[TRELLIS] WARNING: base.py patch anchor not found (upstream changed?)")

    # Shim kaolin (only used for an optional validation path we do not exercise).
    if "kaolin" not in sys.modules:
        _kaolin = types.ModuleType("kaolin")
        _kaolin_utils = types.ModuleType("kaolin.utils")
        _kaolin_testing = types.ModuleType("kaolin.utils.testing")
        def _skip(*a, **k):
            return None
        _kaolin_testing.check_tensor = _skip
        _kaolin_utils.testing = _kaolin_testing
        _kaolin.utils = _kaolin_utils
        sys.modules["kaolin"] = _kaolin
        sys.modules["kaolin.utils"] = _kaolin_utils
        sys.modules["kaolin.utils.testing"] = _kaolin_testing
        print("[TRELLIS] kaolin shim installed")

    from src.engines.trellis_engine import TrellisEngine

    tl_engine = TrellisEngine(device="cuda")
    load_and_time(tl_engine, "TRELLIS")
    runner, df = run_one(tl_engine, "TRELLIS", "/kaggle/working/outputs/trellis")
    display(df)
    del runner, tl_engine
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
except Exception as e:
    print(f"[TRELLIS] Failed with {type(e).__name__}: {e}")
    traceback.print_exc()


In [ ]:
# --- Compare this single-object run against the published 400-run table ------
import glob, json, os
import pandas as pd

REFERENCE = {
    "TripoSR": {
        "latency_s": 3.3793,
        "chamfer_distance": 0.077554,
        "f_score_0.1": 0.94489,
        "psnr": 20.924,
        "vram_peak_gb": 2.32,
        "mesh_size_mb": 1.9818
    },
    "LGM": {
        "latency_s": 9.2801,
        "chamfer_distance": 0.074832,
        "f_score_0.1": 0.93772,
        "psnr": 20.1116,
        "vram_peak_gb": 5.19,
        "mesh_size_mb": 10.19
    },
    "SPAR3D": {
        "latency_s": 37.6326,
        "chamfer_distance": 0.062637,
        "f_score_0.1": 0.96811,
        "psnr": 21.1068,
        "vram_peak_gb": 9.17,
        "mesh_size_mb": 0.5944
    },
    "TRELLIS": {
        "latency_s": 13.4407,
        "chamfer_distance": 0.050176,
        "f_score_0.1": 0.98907,
        "psnr": 21.8531,
        "vram_peak_gb": 5.86,
        "mesh_size_mb": 10.6071
    }
}

rows = []
for p in sorted(glob.glob("/kaggle/working/outputs/*_results.json")):
    try:
        for r in json.load(open(p)):
            rows.append(r)
    except Exception as e:
        print("skip", p, str(e)[:80])

if not rows:
    print("No result rows found — check the engine cells for errors above.")
else:
    live = pd.DataFrame(rows)
    live = live[live["sample_id"] == 'Dino_3']

    # vertex_count / face_count / mesh_size_mb are NOT top-level columns in a
    # current run: the runner writes them inside the `metadata` JSON, while the
    # frozen CSV has them flattened (older schema). Lift them out so the
    # comparison below can see them.
    if "metadata" in live.columns:
        import json as _json
        def _lift(s, k):
            try:
                return _json.loads(s).get(k) if isinstance(s, str) else None
            except Exception:
                return None
        for _k in ("vertex_count", "face_count"):
            live[_k] = live["metadata"].apply(lambda s, k=_k: _lift(s, k))
        # Derive mesh size (MB) from the exported OBJ so it is comparable to the
        # frozen CSV, which recorded the on-disk file size.
        try:
            import glob as _g
            sizes = {}
            for _f in _g.glob("/kaggle/working/outputs/*/qualitative/*/*_mesh.obj"):
                _eng = _f.split("/outputs/")[1].split("/")[0]
                sizes[_eng] = round(os.path.getsize(_f) / (1024 * 1024), 4)
            live["mesh_size_mb"] = live["engine"].str.lower().map(sizes)
        except Exception as _e:
            print("mesh size derivation skipped:", str(_e)[:60])
    print("=" * 78)
    print(f"LIVE RUN — Dino_3")
    print("=" * 78)
    show = [c for c in ["engine", "latency_s", "vram_peak_gb", "chamfer_distance",
                        "f_score_0.1", "f_score_0.2", "psnr", "lpips", "mesh_size_mb"]
            if c in live.columns]
    display(live[show])

    # --- load / e2e timings (not in the paper) -------------------------------
    if TIMINGS:
        print()
        print("=" * 78)
        print("TIMING BREAKDOWN (measured here; the paper reports only latency_s)")
        print("=" * 78)
        tdf = pd.DataFrame(TIMINGS)
        display(tdf)

    # --- delta vs published -------------------------------------------------
    print()
    print("=" * 78)
    print("DELTA vs PUBLISHED (same object, same pinned code)")
    print("=" * 78)
    cmp_rows = []
    for _, r in live.iterrows():
        eng = r["engine"]
        if eng not in REFERENCE:
            continue
        for metric, pub in REFERENCE[eng].items():
            if metric not in live.columns:
                continue
            got = r[metric]
            if pd.isna(got):
                continue
            delta = float(got) - float(pub)
            tol = 0.05 * max(abs(float(pub)), 1e-9)
            cmp_rows.append({
                "engine": eng, "metric": metric,
                "published": round(float(pub), 6),
                "live": round(float(got), 6),
                "delta": round(delta, 6),
                "match": "OK" if abs(delta) <= tol else "DIFF",
            })
    if cmp_rows:
        cdf = pd.DataFrame(cmp_rows)
        display(cdf)
        bad = cdf[cdf["match"] == "DIFF"]
        print()
        if bad.empty:
            print("✅ All comparable metrics within 5% of the published values.")
        else:
            print(f"⚠️  {len(bad)} metric(s) differ by more than 5%:")
            print(bad.to_string(index=False))
            print()
            print("Most likely causes, in order:")
            print("  1. CD/F@0.1 — unseeded surface sampling (see")
            print("     audit/REPRODUCIBILITY_2026-11-22.md). Re-runs move these by")
            print("     ~0.8% median and up to ~20% on a single object.")
            print("  2. PSNR/LPIPS — must be scored at 256x256 (512 render Lanczos-")
            print("     downsampled). Comparing at 512 biases all rows +0.1-0.3 dB.")
            print("  3. Different code_sha than the pin — check outputs/code_sha.txt")
            print("  4. Genuine GPU nondeterminism across T4 instances.")

    # --- expected direction sanity -----------------------------------------
    print()
    print("Sanity: TRELLIS should have the LOWEST CD and HIGHEST F@0.1 / PSNR on")
    print("this object; TripoSR should have the LOWEST latency_s.")


In [ ]:
import shutil, os
from pathlib import Path

WORK = Path("/kaggle/working")
KEEP = {"outputs"}
if WORK.exists():
    for p in WORK.iterdir():
        if p.name in KEEP or p.name.startswith("."):
            continue
        try:
            shutil.rmtree(p) if p.is_dir() else p.unlink()
        except Exception as e:
            print("skip", p.name, str(e)[:60])
print("working dir trimmed; outputs/ preserved")
